In [ ]:
import polars as pl
import numpy as np

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
df = pl.read_csv("../data/stockdata3.csv")

In [ ]:
stocks = [c for c in df.columns if c != "day" and c != "timestr"]
returns = [f"r{s}" for s in stocks]

In [ ]:
INTRA = 0
EXTRA = 1
CLOSE = 2


def get_ret(df, s):
    return (
        df.sort("datetime")
        .fill_nan(None)
        .select(
            [
                pl.col("day"),
                pl.col("datetime"),
                ((pl.col(s) / pl.col(s).shift(1)).log(base=10) * 1e4).alias(f"r{s}"),
            ]
        )
        .with_columns(
            [
                pl.when(pl.col("day").diff() == 0)
                .then(pl.lit(INTRA))
                .otherwise(pl.lit(EXTRA))
                .alias("return_type")
            ]
        )
    )


def get_ret_cc(df, s):
    return (
        df.sort("datetime")
        .fill_nan(None)
        .group_by("day")
        .agg([pl.col(s).drop_nulls().last()])
        .sort("day")  # must sort before shift(1), group_by output is unordered
        .with_columns(
            [
                ((pl.col(s) / pl.col(s).shift(1)).log() * 1e4).alias(f"r{s}"),
                pl.lit(CLOSE).alias("return_type"),
            ]
        )
    )


def get_ret_all(df, s):
    ret = get_ret(df, s[0])
    for stock in s[1:]:
        print(stock)
        ret = ret.join(get_ret(df, stock), on="datetime", how="inner")

    return ret


# def add_ret(df):
#     return (
#         df.sort("datetime")
#         .fill_nan(None)
#         .with_columns(
#             [
#                 ((pl.col(s) / pl.col(s).shift(1)).log() * 1e4).alias(f"r{s}")
#                 for s in stocks
#             ]
#         )
#         .with_columns(
#             [
#                 pl.when(pl.col("day").diff() == 0)
#                 .then(pl.lit(INTRA))
#                 .otherwise(pl.lit(EXTRA))
#                 .alias("return_type")
#             ]
#         )
#     )

In [ ]:
def daily_vol_from_ret(df, returns):
    if isinstance(returns, str):
        return df.group_by("day").agg(pl.col(returns).pow(2).sum().sqrt())
    else:
        return df.group_by("day").agg(
            [
                (pl.col(s1) * pl.col(s2)).sum().sqrt().alias(f"")
                for s1 in returns
                for s2 in returns
            ]
        )


def daily_vol(df, stocks, return_type=None):

    mask = (
        pl.col("return_type") == return_type
        if return_type is not None
        else pl.lit(True)
    )
    if isinstance(stocks, str):
        stocks = [stocks]
    returns = [f"r{s}" for s in stocks]
    vols = daily_vol_from_ret(get_ret(df, stocks[0]).filter(mask), returns[0])
    for s, r in zip(stocks[1:], returns[1:]):
        vols = vols.join(daily_vol_from_ret(get_ret(df, s).filter(mask), r), on="day")
    return vols

In [ ]:
# deal with c: likely split
def normalize_c(df):
    idxs = df.filter(pl.col("c").diff().abs() > 0.4 * pl.col("c"))["index"]
    if len(idxs) > 0:
        index = idxs[0]
    else:
        return df
    print(index)
    return df.with_columns(
        pl.when(pl.col("index") < index)
        .then(pl.col("c") / 2)
        .otherwise(pl.col("c"))
        .alias("c")
    )

In [ ]:
# deal with c: likely split
# deal with c: likely split
def mask_micro_noise(col, thd=100, neighbor=1):
    ratio = thd / 1e4  # bps
    ret = (pl.col(col) / pl.col(col).shift(1)).log()
    mask = pl.lit(False)
    for n in range(1, neighbor + 1):
        mask |= (
            (ret * ret.shift(-n) < 0)
            & (ret.abs() > ratio)
            & (ret.shift(-n).abs() > ratio)
        )
    return mask


def mask_denoised(col, thd=100, neighbor=1):
    return ~mask_micro_noise(col, thd, neighbor)


# def clean_d(df):
#     return df

In [ ]:
def downsample(df, freq):
    return df.group_by_dynamic("datetime", every=freq, group_by="day").agg(
        pl.all().last()
    )

## load data

In [ ]:
df = (
    df.with_columns(
        (pl.date(2000, 1, 1) + pl.duration(days=pl.col("day") - 1))
        .dt.combine(pl.col("timestr").str.to_time("%H:%M:%S"))
        .alias("datetime")
    )
    if "datatime" not in df.columns
    else df
)
display(df.select("day", "timestr", "datetime"))
df = df.with_row_index() if "index" not in df.columns else df

In [ ]:
# clean a and d
df = df.with_columns(
    pl.when(pl.col("a") == 0).then(None).otherwise(pl.col("a")).alias("a"),
    pl.when(pl.col("d") == 1).then(None).otherwise(pl.col("d")).alias("d"),
)
# clean c
df = normalize_c(df)
# filter d
NOISE_THD = 100  # 100 bps
df = df.filter(mask_denoised("d", NOISE_THD))

In [ ]:
# categories:
stocks_normal = ["a", "c", "e"]  # leave it as is
stocks_downsample_5min = ["d"]  # down sample to 5min
stocks_downsample_120min = ["b"]  # down sample to 120min
stocks_jump = ["f"]  # special treat, do Poisson jump or monthly close-close volatility

In [ ]:
display(downsample(df, "10m"))

In [ ]:
df

In [ ]:
vols = daily_vol(df, stocks_normal + stocks_jump, return_type=INTRA)
vols = vols.join(
    daily_vol(downsample(df, "5m"), stocks_downsample_5min, return_type=INTRA), on="day"
)
vols = vols.join(
    daily_vol(downsample(df, "120m"), stocks_downsample_120min, return_type=INTRA),
    on="day",
)

# close-to-close daily returns, one row per day --- join on day (not concat:
# each get_ret_cc frame is 1-row-per-day, a horizontal concat would misalign).
ret_cc = get_ret_cc(df, stocks[0]).select("day", f"r{stocks[0]}")
for s in stocks[1:]:
    ret_cc = ret_cc.join(get_ret_cc(df, s).select("day", f"r{s}"), on="day")
ret_cc = ret_cc.sort("day")

In [ ]:
ret_cc

In [ ]:
# vols_bm = daily_vol(df, stocks_normal + stocks_jump)
# vols_bm = vols_bm.join(daily_vol(df, stocks_downsample), on="day")

In [ ]:
vols = vols.sort("day")
# vols_bm = vols_bm.sort("day")

In [ ]:
for v in returns:
    # print(f"=============={v}==============")
    plt.step(vols["day"], vols[v], label=v)
    # plt.plot(vols_bm[v])
plt.legend()
plt.show()

## Check autocorrelation

In [ ]:
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

acfs, pacfs = {}, {}
for r in returns:
    data = vols[r].drop_nulls().to_numpy()
    acfs[r] = acf(data, nlags=40, alpha=0.05)
    pacfs[r] = pacf(data, nlags=40, alpha=0.05)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))
    plot_acf(data, lags=40, alpha=0.05, ax=ax1)
    plot_pacf(data, lags=40, alpha=0.05, ax=ax2)
    ax1.set_title(f"ACF {r}")
    ax2.set_title(f"PACF {r}")
    plt.tight_layout()
    plt.show()

## Check daily volatilities

In [ ]:
for r in returns:
    print(f"====================={r}=====================")
    print(vols[r].mean(), vols[r].std())
    plt.step(vols["day"], vols[r])
    plt.title(r)
    plt.show()

## Check the short day

In [ ]:
print(vols.filter(pl.col("day") == 327))
print(vols.mean())

In [ ]:
np.sqrt(391 / 211)

## Check lagged cross correlation

In [ ]:
for lag in [1, 5, 21]:
    lagged = (
        vols.select(returns).shift(lag).rename({c: f"{c}_lag{lag}" for c in returns})
    )
    combined = pl.concat([vols.select(returns), lagged], how="horizontal").drop_nulls()
    corr = combined.corr()
    n = len(returns)
    block = corr[:n, n:]
    block = block.rename({f"{c}_lag{lag}": c for c in returns})
    block = block.insert_column(0, pl.Series("name", returns))
    print(f"\ncorr(RV_i(t), RV_j(t-{lag})):")
    print(block)

## Model fitting/prediction

In [ ]:
from collections.abc import Callable, Sequence


# ---- pooling: aggregate the daily-RV expr over a window of w trading days ----
# Daily RV is itself sqrt(daily variance); the choice is how those vols compose.
def pool_mean(e: pl.Expr, w: int) -> pl.Expr:
    """Plain average of daily RV over w days --- the literal Corsi HAR pooling."""
    return e.rolling_mean(w)


def pool_rms(e: pl.Expr, w: int) -> pl.Expr:
    """RMS of daily RV over w days: sqrt(mean RV^2). Dimensionally consistent
    (variances are additive, vols are not); stays on the daily-vol scale."""
    return e.pow(2).rolling_mean(w).sqrt()


# ---- feature builders: act on the daily-RV expr, return a causal (info<=t) expr ----
def ewma(half_life: float, on_variance: bool = False):
    """Exponentially weighted moving avg of daily RV, info up to row t (causal).

    A self-contained causal feature --- pass it in `feature_fns`.
    half_life : decay in trading days. RiskMetrics lambda=0.94 ~= half_life 11.
    on_variance : if True, EWMA the variance (RV^2) RiskMetrics-style and sqrt
                  back; if False, EWMA the vol series directly.
    """

    def fn(e: pl.Expr) -> pl.Expr:
        if on_variance:
            return e.pow(2).ewm_mean(half_life=half_life, adjust=False).sqrt()
        return e.ewm_mean(half_life=half_life, adjust=False)

    return fn


def make_features_targets(
    vols: pl.DataFrame,
    cols: Sequence[str],
    windows: Sequence[int] = (1, 5, 21),
    horizon: int = 21,
    pool: Callable[[pl.Expr, int], pl.Expr] = pool_rms,
    feature_fns: dict[str, Callable[[pl.Expr], pl.Expr]] | None = None,
    target_fn: Callable[[pl.Expr], pl.Expr] | None = None,
    day_col: str = "day",
    drop_nulls: bool = False,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Turn a daily realized-vol frame into (features, target) frames.

    `vols` has one row per trading day plus `day_col`. Rows are assumed to be
    consecutive trading days, so a window / horizon of N rows == N trading days.
    All features use info up to and including day t; the target is strictly
    forward (t+1..t+h), so there is no overlap and no leakage. For row t:
      HAR feature {c}_avg{w} = pool of daily RV over the trailing w days [t-w+1..t]
      extra       {c}{name}  = feature_fns[name](col c)   (causal)
      target      {c}_fwd    = pool of daily RV over the forward h days [t+1..t+h]

    windows : HAR averaging windows in trading days. (1,5,21) are the classic
              daily / weekly / monthly HAR-RV components (Corsi 2009 uses 22).
              w=1 is RV(t) itself and keeps the bare name {c}.
    horizon : 21 is the usual "1 month" (252 trading days / 12 ~= 21); Corsi's
              HAR-RV uses 22; 20 is just a round "4 weeks". Default 21.
    pool    : (expr, w)->expr aggregator used for BOTH the HAR features and the
              default target, so the two always share one convention. pool_rms
              (default) is dimensionally consistent; pool_mean is literal Corsi.
    feature_fns : optional {name: fn(expr)->expr} extra causal features, e.g.
              {"_ewma11": ewma(11)}. Applied per column, info up to t.
    target_fn : optional override of the forward target. Default is
              pool(., horizon) shifted back h rows. Must be forward-looking.
    drop_nulls : if False (default) keep every row from day 1 --- partial
              windows (head) and missing forward targets (tail) stay null, so
              you can experiment with models that don't need a full HAR window.
              Set True for a ready-to-fit frame with the head/tail trimmed.

    Returns (X, y), both keyed by `day_col` and row-aligned.
    """
    vols = vols.sort(day_col)
    target_fn = target_fn or (lambda e: pool(e, horizon).shift(-horizon))

    feature_exprs = [
        pool(pl.col(c), w).alias(f"{c}" + (f"_avg{w}" if w != 1 else ""))
        for c in cols
        for w in windows
    ]
    if feature_fns:
        feature_exprs += [
            fn(pl.col(c)).alias(f"{c}{name}")
            for name, fn in feature_fns.items()
            for c in cols
        ]
    target_exprs = [target_fn(pl.col(c)).alias(f"{c}_fwd") for c in cols]

    combined = vols.select(pl.col(day_col), *feature_exprs, *target_exprs)
    if drop_nulls:
        combined = combined.drop_nulls()  # trims the window head + horizon tail

    feat_names = [e.meta.output_name() for e in feature_exprs]
    tgt_names = [e.meta.output_name() for e in target_exprs]
    return combined.select(day_col, *feat_names), combined.select(day_col, *tgt_names)

In [ ]:
X, y = make_features_targets(
    vols,
    returns,
    windows=[1, 5, 21],  # HAR: daily / weekly / monthly pooled vol
    horizon=21,
    pool=pool_rms,  # pool_mean for the literal Corsi HAR average
    feature_fns={"_ewma11": ewma(11), "_ewma44": ewma(44)},
)
print("X:", X.shape, " y:", y.shape)
display(X.head())
display(y.head())

In [ ]:
X.sort("day")

### GARCH benchmark

In [ ]:
from arch import arch_model


def fit_garch(returns, p=1, q=1):
    model = arch_model(returns, vol="Garch", p=p, q=q, rescale=True)
    res = model.fit(disp="off")
    print(res.summary())
    return res


def garch_predict(res):
    forecast = res.forecast(horizon=21, reindex=False)
    # Sum daily variances over 21 days for monthly variance
    monthly_var = forecast.variance.iloc[-1].sum()
    return np.sqrt(monthly_var)
    # monthly_vol_ann = np.sqrt(monthly_var * 252 / 21) / 100  # if rescaled to pct
    # print(f"Annualized vol: {monthly_vol_ann:.2%}")

In [ ]:
def garch_features(df, stocks, horizon=21, day_col="day"):
    """Per-day GARCH(1,1) features --- one fit per stock on close-to-close
    daily returns. Returns a frame keyed by `day_col`; per stock s:
      r{s}_garch     : GARCH conditional vol for day t   (filtered estimate)
      r{s}_garch_fwd : GARCH forecast of avg daily vol over days t+1..t+horizon

    Not a make_features_targets feature_fn: GARCH needs an MLE fit and a
    recursive variance filter, neither of which is a vectorized polars expr ---
    so compute it separately and join the result onto X.

    Caveat: ONE fit on the full sample, so the *parameters* see all history.
    Fine for exploration; for an honest backtest refit on a rolling window
    (the conditional-vol path is itself causal). Returns use log10*1e4, the
    get_ret / vols convention, so r{s}_garch is comparable to the vols frame.
    """
    daily = (
        df.sort("datetime")
        .group_by(day_col, maintain_order=True)
        .agg([pl.col(s).drop_nulls().last() for s in stocks])
        .sort(day_col)
    )
    out = {day_col: daily[day_col]}
    for s in stocks:
        r = ((daily[s] / daily[s].shift(1)).log() * 1e4).to_numpy()
        res = arch_model(r[1:], vol="Garch", p=1, q=1, rescale=True).fit(disp="off")
        sc = res.scale  # arch may rescale the returns internally; undo it
        fc = res.forecast(horizon=horizon, start=0, reindex=True)
        fwd = np.sqrt(fc.variance.to_numpy().mean(axis=1)) / sc
        out[f"r{s}_garch"] = np.r_[np.nan, res.conditional_volatility / sc]
        out[f"r{s}_garch_fwd"] = np.r_[np.nan, fwd]
    return pl.DataFrame(out)

In [ ]:
garch = garch_features(df, stocks, horizon=21)
display(garch.head())

# join GARCH columns onto X (guarded so re-running this cell stays idempotent)
if not any(c.endswith("_garch") for c in X.columns):
    X = X.join(garch, on="day", how="left")
print("X with GARCH:", X.shape)